<a href="https://colab.research.google.com/github/rakshansingh12/protein-function-classification-ml/blob/main/02_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install biopython pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 45.4 MB/s eta 0:00:00


In [3]:
from Bio import SeqIO
from Bio.SeqUtils import ProtParam
import pandas as pd

In [4]:
enzyme_records = list(SeqIO.parse("/enzymes_clean.fasta", "fasta"))
non_enzyme_records = list(SeqIO.parse("/non_enzymes_clean.fasta", "fasta"))

print("Enzymes:", len(enzyme_records))
print("Non-enzymes:", len(non_enzyme_records))


Enzymes: 120035
Non-enzymes: 120035


count of the clean enzymes and non enzymes

In [5]:
amino_acids = list("ACDEFGHIKLMNPQRSTVWY")


In [6]:
def aa_composition(sequence):
    seq = str(sequence)
    length = len(seq)
    return {aa: seq.count(aa) / length for aa in amino_acids}

1. Counts how often each amino acid appears
2. Divides by sequence length
3. Produces normalized frequencies

This avoids bias from protein length.



In [7]:
def physchem_features(sequence):
    analyzer = ProtParam.ProteinAnalysis(str(sequence))
    return {
        "length": len(sequence),
        "molecular_weight": analyzer.molecular_weight(),
        "isoelectric_point": analyzer.isoelectric_point(),
        "aromaticity": analyzer.aromaticity(),
        "gravy": analyzer.gravy()
    }


define a function called physchem_features and analyzer acts as a biochem calculatorwhich internally knows Amino Acid weights, pKa values, hydrophobicity scales etc.

In [8]:
def extract_features(record):
    features = {}
    features.update(aa_composition(record.seq))
    features.update(physchem_features(record.seq))
    return features

combine features for one protein i.e one feature dictionary per protein.

In [9]:
type(enzyme_records)


list

In [10]:
import re

def has_ambiguous(seq):
    return bool(re.search(r"[XBZJUO]", str(seq)))


In [11]:
enzyme_records = [r for r in enzyme_records if not has_ambiguous(r.seq)]
non_enzyme_records = [r for r in non_enzyme_records if not has_ambiguous(r.seq)]

print("After removing ambiguous + O:")
print("Enzymes:", len(enzyme_records))
print("Non-enzymes:", len(non_enzyme_records))


After removing ambiguous + O:
Enzymes: 120014
Non-enzymes: 120035


In [12]:
n = min(len(enzyme_records), len(non_enzyme_records))
n


120014

In [13]:
enzyme_records = enzyme_records[:n]
non_enzyme_records = non_enzyme_records[:n]


In [14]:
print("Enzymes:", len(enzyme_records))
print("Non-enzymes:", len(non_enzyme_records))

Enzymes: 120014
Non-enzymes: 120014


To avoid class imbalance, the dataset was balanced by undersampling the majority class so that equal numbers of enzyme and non-enzyme proteins were used for model training

In [15]:
data = []

for r in enzyme_records:
    feats = extract_features(r)
    feats["label"] = 1  # enzyme
    data.append(feats)

for r in non_enzyme_records:
    feats = extract_features(r)
    feats["label"] = 0  # non-enzyme
    data.append(feats)


This builds feature table


In [16]:
df = pd.DataFrame(data)
df.head()

,A,C,D,E,F,G,H,I,K,L,...,T,V,W,Y,length,molecular_weight,isoelectric_point,aromaticity,gravy,label
0,0.115523,0.018051,0.046931,0.079422,0.025271,0.115523,0.036101,0.032491,0.025271,0.104693,...,0.054152,0.108303,0.000000,0.021661,277,28955.5180,5.016803,0.046931,0.114079,1
1,0.127193,0.010965,0.063596,0.063596,0.039474,0.078947,0.019737,0.028509,0.024123,0.114035,...,0.052632,0.054825,0.013158,0.019737,456,50118.1914,5.703872,0.072368,-0.308333,1
2,0.120482,0.008032,0.052209,0.076305,0.008032,0.076305,0.024096,0.052209,0.016064,0.108434,...,0.060241,0.092369,0.008032,0.008032,249,27001.5057,5.187092,0.024096,-0.118876,1
3,0.111782,0.027190,0.045317,0.093656,0.015106,0.069486,0.021148,0.033233,0.012085,0.102719,...,0.066465,0.069486,0.015106,0.018127,331,36462.0996,4.925406,0.048338,-0.217221,1
4,0.113208,0.010782,0.053908,0.067385,0.018868,0.105121,0.024259,0.048518,0.026954,0.113208,...,0.051213,0.091644,0.010782,0.016173,371,39592.7800,5.657662,0.045822,0.011590,1


I converted extracted protein sequence features into a structured pandas DataFrame, where each row represents a protein and columns represent amino acid composition, physicochemical properties, and class labels.

In [17]:
df.shape


(240028, 26)

sanity check

In [18]:
df.isnull().sum()

,0
A,0
C,0
D,0
E,0
F,0
G,0
H,0
I,0
K,0
L,0


check for missing values.

In [19]:
df.to_csv("/content/protein_features.csv", index=False)

I extracted amino acid composition and physicochemical properties from protein sequences to create numerical feature representations suitable for classical machine learning models.